In [26]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [27]:
import sys
print(sys.executable)

d:\LLM\trust-behaviours\.venv\Scripts\python.exe


In [28]:
from dotenv import load_dotenv
load_dotenv()

import kagglehub
import pandas as pd
import os
from torch.utils.data import Dataset, DataLoader, random_split
from torch import nn, optim
import torch
from tqdm import tqdm

EMBED_DIM = 128
HIDDEN_DIM = 64
MAX_EPOCHS = 6

VOCAB_SIZE = 20000
MIN_OCCURRENCE = 23

path = kagglehub.dataset_download("snap/amazon-fine-food-reviews")
df = pd.read_csv(os.path.join(path, "Reviews.csv"), usecols=["Text", "Score"])

In [29]:
df["Text"] = df["Text"].str.lower()
df["Text"] = df["Text"].str.replace(r"<[^>]+>", " ", regex=True)
df["Text"] = df["Text"].str.replace(r"[^\w\s]", " ", regex=True)

In [30]:
# Build vocabulary
from collections import Counter
word_freq = Counter(" ".join(df["Text"]).split())
most_common = word_freq.most_common(VOCAB_SIZE)

# Tokens spéciaux
tokens_list = {"<PAD>": 0, "<UNK>": 1}

for word, _ in most_common:
    tokens_list[word] = len(tokens_list)

In [31]:
def tokenize(text, max_length=222):
    sentence = [tokens_list.get(word, tokens_list["<UNK>"]) for word in text.split()]
    # Pad or truncate the sentence to the maximum length
    if len(sentence) < max_length:
        sentence.extend([tokens_list["<PAD>"]] * (max_length - len(sentence)))
    else:
        sentence = sentence[:max_length]
    return sentence

df["tokens"] = df["Text"].apply(tokenize)

In [32]:
class ReviewsDataset(Dataset):
    def __init__(self, df):
        self.X = torch.tensor(df["tokens"].tolist(), dtype=torch.long)
        self.Y = torch.tensor(df["Score"].tolist(), dtype=torch.long) - 1  # Scores de 1 à 5 -> labels de 0 à 4

    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

In [33]:
class FeelingModel(nn.Module):
    def __init__(self, vocab_size, embed_dim = 256, hidden_dim = 128, output_dim = 5):
        super(FeelingModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=2, bidirectional=True, dropout=0.3)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, output_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        output = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return self.fc(output)

In [34]:
train_size = int(0.7 * len(df))
val_size = int(0.15 * len(df))
test_size = len(df) - train_size - val_size

dataset = ReviewsDataset(df)
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size], 
                                                        generator=torch.Generator().manual_seed(42))

In [35]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

In [36]:
device = torch.device("cpu")
if torch.cuda.is_available():
  device = torch.device("cuda")
elif torch.backends.mps.is_available():
  device = torch.device("mps")

print(device)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA device")

model = FeelingModel(vocab_size=len(tokens_list), embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

cuda
NVIDIA GeForce RTX 4060 Ti


In [37]:
previous_val_loss = float('inf')
for epoch in range(MAX_EPOCHS):
    # Entrainement
    model.train()
    for X_batch, Y_batch in tqdm(train_loader, desc=f"Epoch {epoch+1} - Training"):
        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)

        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, Y_batch)
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    with torch.no_grad():
        for X_batch, Y_batch in tqdm(val_loader, desc=f"Epoch {epoch+1} - Validation"):
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)
            output = model(X_batch)
            val_loss_batch = criterion(output, Y_batch)   # ← calcul réel
            predicted_value = output.argmax(dim=1)
            correct += (Y_batch == predicted_value).sum().item()
            total += Y_batch.size(0)
            val_loss += val_loss_batch.item()
    val_accuracy = correct / total
    val_loss /= len(val_loader)

    print(f"Epoch {epoch+1}/{MAX_EPOCHS} | val_loss: {val_loss:.4f} | val_accuracy: {val_accuracy:.4f} | lr: {optimizer.param_groups[0]['lr']:.6f}")
    if (previous_val_loss - val_loss) < 0.005:
        break
    previous_val_loss = val_loss

Epoch 1 - Training:   0%|          | 0/6218 [00:00<?, ?it/s]


RuntimeError: mat1 and mat2 shapes cannot be multiplied (222x5 and 64x5)

In [ ]:
torch.save(model.state_dict(), "../model.pt")